# 10_02 — 장문 열화 진단: 표현(pooling) vs 결정(threshold)

**목적.** 문제 2번(maxlen=8192에서도 긴 문서 성능 저하 → label-aware attention으로 헤드를 교체)에 GPU를 쓰기 전, 저하의 **성격**을 저장된 로짓만으로(로컬 CPU) 판별하는 게이트다. 이전 무훈련 게이트(오라클-Lno·오라클-τ·오라클-k)와 같은 역할 — 비싼 실험 전에 헤드룸과 그 도달 가능성을 먼저 잰다.

두 질문에 답한다.

1. **독립성** — 장문 저하가 이미 닫힌 카디널리티 축(k≥2 과소예측, [ADR-0009])의 위장인가? 긴 청구항 → 넓은 범위 → 다중 라벨이라면 문제 2번은 문제 1번이다. 길이×k 분해로 가른다.
2. **표현 vs 결정** — 저하가 (a) 풀링 병목으로 정답 신호가 랭킹에서 사라지는 표현 붕괴인가, (b) 정보는 랭킹에 살아 있고 top-1/임계 배치만 미끄러지는 결정층 문제인가. label-aware attention은 (a)만 고친다. threshold-free 랭킹 지표(R-Precision·P@N)의 길이별 거동으로 가른다.

로짓은 풀링 **하류**라 (a)/(b)를 완전히 분리하진 못한다 — 확정에는 8192 hidden-state 덤프가 필요하다. 이 게이트는 그 GPU 실험의 **기대 이득 상한**을 경계짓는다.

대상 = 운영 채택 exp1(A.X 8192, 최고 micro 0.8685) + exp2(A.X 512) 창-대조. 판정 축 = 멀티라벨 micro-F1 · P@1 · R-Precision(`PROJECT.md` 평가 절).

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import numpy as np

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
os.environ["HF_HOME"] = str(ROOT / ".hf_cache")

from datasets import load_dataset
from sklearn.metrics import f1_score
# 오류 분석 하니스 재사용(src/error_analysis.py — editable 설치 최상위 모듈)
from error_analysis import build_gold, r_precision

In [2]:
config = {
    "num_labels": 188,
    "tau": 0.5,
    "raw_ds": "ingyoun/patent-clean-text",
    "out_path": ROOT / "output",
}
NUM, TAU, OUT = config["num_labels"], config["tau"], config["out_path"]
BINS = ["<=512", "512-1024", "1024-2048", ">2048"]   # kobert_len 축 — modernbert-comparison.md B0~B3
# exp1=운영 채택 8192, exp2=512 창-대조
MODELS = {"exp1_len8192": "modernbert-patent-len8192", "exp2_len512": "modernbert-patent-len512"}

In [3]:
# 라벨·길이축·로짓 로딩 + 행 순서 검증
ds = load_dataset(config["raw_ds"], split="test")
doc_ids = json.loads((OUT / "doc_ids_test.json").read_text(encoding="utf-8"))
assert ds["document_id"] == doc_ids, "로짓 행 순서 != 데이터셋 행 순서"

Y = build_gold(ds["label_ids"], len(ds), NUM)   # (N, 188) 멀티핫
k = Y.sum(1)                                     # 문서별 정답 개수
lb = np.array(ds["length_bin"])                  # kobert_len 구간 문자열
N = len(Y)

L = {name: np.load(OUT / f"logits_{tag}_test.npy") for name, tag in MODELS.items()}
for name, arr in L.items():
    assert arr.shape == (N, NUM), (name, arr.shape)

print(f"N={N:,}  k>=2={ (k>=2).mean():.1%}")
print("bin 분포:", {b: int((lb == b).sum()) for b in BINS})

N=11,271  k>=2=15.0%
bin 분포: {'<=512': 3197, '512-1024': 5183, '1024-2048': 2342, '>2048': 549}


In [4]:
# 지표 헬퍼 — pred = logit>=0 (τ=0.5). 랭킹 지표는 로짓 순서=확률 순서(sigmoid 단조)라 로짓에 직접 적용.
def micro(Ys, pred):    return float(f1_score(Ys, pred, average="micro",   zero_division=0))
def sample_f1(Ys, pred):return float(f1_score(Ys, pred, average="samples", zero_division=0))

def p_at_1(logits, Ys):
    top1 = logits.argmax(1)
    return float(Ys[np.arange(len(Ys)), top1].mean())          # 앵커 top-1 정확도 = 1 - 앵커 오류율

def topN_single(logits, Ys, n):
    # k==1 문서: 유일한 정답이 상위 n 랭크 안에 드는 비율 (threshold-free)
    order = np.argsort(-logits, axis=1)[:, :n]
    gold = Ys.argmax(1)
    return float(np.mean([gold[i] in order[i] for i in range(len(Ys))]))

## 검증 게이트 — 재계산치가 SSOT와 일치하는가

exp1 bin별 micro(`modernbert-comparison.md` 「3-모델 bin 비교」)와 exp1−exp2 델타(「창 확장 signature」)를 재계산해 SSOT와 4자리 대조한다. 일치 = 로짓 행 순서·라벨·길이축 조인이 옳다는 증거.

In [5]:
SSOT_micro  = {"<=512": 0.8765, "512-1024": 0.8734, "1024-2048": 0.8516, ">2048": 0.8490}
SSOT_delta  = {"<=512": 0.46,   "512-1024": 0.73,   "1024-2048": 1.18,   ">2048": 2.64}   # exp1-exp2 pt

p1, p2 = (L["exp1_len8192"] >= 0), (L["exp2_len512"] >= 0)
print(f"{'bin':12s}{'exp1 micro':>12}{'SSOT':>8}{'  exp1-exp2':>12}{'SSOT':>7}")
ok = True
for b in BINS:
    sel = lb == b
    m1, m2 = micro(Y[sel], p1[sel]), micro(Y[sel], p2[sel])
    d = 100 * (m1 - m2)
    a = abs(m1 - SSOT_micro[b]) < 2e-4 and abs(d - SSOT_delta[b]) < 0.02
    ok &= a
    print(f"{b:12s}{m1:>12.4f}{SSOT_micro[b]:>8}{d:>+12.2f}{SSOT_delta[b]:>+7.2f}   {'OK' if a else 'DIFF'}")
assert ok, "SSOT 불일치 — 파이프라인 점검"
print("\nverify 통과: bin별 micro·창 델타 SSOT 4자리 일치")

bin           exp1 micro    SSOT   exp1-exp2   SSOT
<=512             0.8765  0.8765       +0.46  +0.46   OK


512-1024          0.8734  0.8734       +0.73  +0.73   OK
1024-2048         0.8516  0.8516       +1.18  +1.18   OK
>2048             0.8490   0.849       +2.64  +2.64   OK

verify 통과: bin별 micro·창 델타 SSOT 4자리 일치


## 1. 저하 곡선 — exp1 길이대별

exp1 자체가 길이에 따라 어떻게 떨어지는지(micro·sample·P@1·R-Precision·empty), 그리고 창 확장(exp1−exp2)이 어느 구간에서 회복하는지.

In [6]:
print(f"{'bin':12s}{'n':>6}{'e1_micro':>10}{'e2_micro':>10}{'e1-e2pt':>9}"
      f"{'sample':>9}{'P@1':>8}{'R-Prec':>9}{'empty':>8}")
curve = {}
for b in BINS:
    sel = lb == b
    e1, e2 = p1[sel], p2[sel]
    lo1 = L["exp1_len8192"][sel]
    rec = {"n": int(sel.sum()),
           "e1_micro": round(micro(Y[sel], e1), 4), "e2_micro": round(micro(Y[sel], e2), 4),
           "delta_pt": round(100*(micro(Y[sel],e1)-micro(Y[sel],e2)), 2),
           "sample_f1": round(sample_f1(Y[sel], e1), 4),
           "p_at_1": round(p_at_1(lo1, Y[sel]), 4),
           "r_precision": round(float(r_precision(lo1, Y[sel]).mean()), 4),
           "empty_rate": round(float((e1.sum(1) == 0).mean()), 4)}
    curve[b] = rec
    print(f"{b:12s}{rec['n']:>6}{rec['e1_micro']:>10.4f}{rec['e2_micro']:>10.4f}{rec['delta_pt']:>+9.2f}"
          f"{rec['sample_f1']:>9.4f}{rec['p_at_1']:>8.4f}{rec['r_precision']:>9.4f}{rec['empty_rate']:>8.4f}")
print("\n저하 실재: exp1 micro 0.8765→0.8490 · P@1 0.9115→0.8689 · R-Prec 0.9039→0.8621")
print("창 확장(exp1-exp2)은 최장 문서에서 최대(+2.64pt) — 8192 창이 장문 컨텍스트 가치를 이미 회수")

bin              n  e1_micro  e2_micro  e1-e2pt   sample     P@1   R-Prec   empty


<=512         3197    0.8765    0.8719    +0.46   0.8895  0.9115   0.9039  0.0144


512-1024      5183    0.8734    0.8661    +0.73   0.8887  0.9095   0.9020  0.0120
1024-2048     2342    0.8516    0.8398    +1.18   0.8669  0.8958   0.8849  0.0132


>2048          549    0.8490    0.8226    +2.64   0.8512  0.8689   0.8621  0.0237

저하 실재: exp1 micro 0.8765→0.8490 · P@1 0.9115→0.8689 · R-Prec 0.9039→0.8621
창 확장(exp1-exp2)은 최장 문서에서 최대(+2.64pt) — 8192 창이 장문 컨텍스트 가치를 이미 회수


## 2. 카디널리티와 독립인가

문제 2번이 닫힌 문제 1번(k≥2 과소예측)의 위장인지 가른다. (i) k≥2 비율이 길이에 따라 오르는가, (ii) **k=1만** 봐도(카디널리티 완전 통제) 저하가 살아남는가.

In [7]:
print("길이 bin별 카디널리티 구성:")
kx = {}
for b in BINS:
    sel = lb == b
    kx[b] = {"frac_k>=2": round(float((k[sel] >= 2).mean()), 3), "mean_k": round(float(k[sel].mean()), 3)}
    print(f"  {b:12s} k>=2 비율 {kx[b]['frac_k>=2']:.3f}   평균 k {kx[b]['mean_k']:.3f}")

print("\nk==1 문서만(카디널리티 통제):")
k1 = {}
for b in BINS:
    sel = (lb == b) & (k == 1)
    lo = L["exp1_len8192"][sel]
    k1[b] = {"n": int(sel.sum()), "micro": round(micro(Y[sel], p1[sel]), 4), "p_at_1": round(p_at_1(lo, Y[sel]), 4)}
    print(f"  {b:12s} n={k1[b]['n']:5d}  micro={k1[b]['micro']:.4f}  P@1={k1[b]['p_at_1']:.4f}")

print("\nk>=2 문서만:")
k2 = {}
for b in BINS:
    sel = (lb == b) & (k >= 2)
    lo = L["exp1_len8192"][sel]
    k2[b] = {"n": int(sel.sum()), "micro": round(micro(Y[sel], p1[sel]), 4),
             "r_precision": round(float(r_precision(lo, Y[sel]).mean()), 4)}
    print(f"  {b:12s} n={k2[b]['n']:5d}  micro={k2[b]['micro']:.4f}  R-Prec={k2[b]['r_precision']:.4f}")

print("\n판정: k>=2 비율은 길이와 무관(0.144→0.131, 최장이 오히려 낮음). "
      "k==1만 봐도 P@1 0.9098→0.8616로 저하 생존 → 장문 열화는 카디널리티와 독립인 별개 현상")

길이 bin별 카디널리티 구성:
  <=512        k>=2 비율 0.144   평균 k 1.179
  512-1024     k>=2 비율 0.157   평균 k 1.215
  1024-2048    k>=2 비율 0.148   평균 k 1.211
  >2048        k>=2 비율 0.131   평균 k 1.206

k==1 문서만(카디널리티 통제):
  <=512        n= 2737  micro=0.8995  P@1=0.9098
  512-1024     n= 4369  micro=0.8983  P@1=0.9059
  1024-2048    n= 1996  micro=0.8756  P@1=0.8913
  >2048        n=  477  micro=0.8537  P@1=0.8616

k>=2 문서만:
  <=512        n=  460  micro=0.8068  R-Prec=0.8689
  512-1024     n=  814  micro=0.8089  R-Prec=0.8800
  1024-2048    n=  346  micro=0.7861  R-Prec=0.8479
  >2048        n=   72  micro=0.8353  R-Prec=0.8653

판정: k>=2 비율은 길이와 무관(0.144→0.131, 최장이 오히려 낮음). k==1만 봐도 P@1 0.9098→0.8616로 저하 생존 → 장문 열화는 카디널리티와 독립인 별개 현상


## 3. 표현 붕괴인가, 결정층 미끄러짐인가 (핵심)

k=1 문서(85%, 카디널리티 교락 없음)에서 정답이 상위 N랭크에 드는 비율. **P@1은 떨어지는데 P@3·P@5는 유지**되면 정답은 여전히 랭킹 최상단 근처에 있고(정보가 풀링을 통과해 살아 있음) 미끄러지는 건 top-1/임계 배치 = **결정층**. P@3·P@5까지 무너지면 랭킹 자체가 열화 = **표현/풀링**.

In [8]:
print(f"{'bin':12s}{'n':>6}{'P@1':>9}{'P@3':>9}{'P@5':>9}")
topn = {}
for b in BINS:
    sel = (lb == b) & (k == 1)
    lo, Ys = L["exp1_len8192"][sel], Y[sel]
    topn[b] = {"n": int(sel.sum()),
               "p@1": round(topN_single(lo, Ys, 1), 4),
               "p@3": round(topN_single(lo, Ys, 3), 4),
               "p@5": round(topN_single(lo, Ys, 5), 4)}
    print(f"{b:12s}{topn[b]['n']:>6}{topn[b]['p@1']:>9.4f}{topn[b]['p@3']:>9.4f}{topn[b]['p@5']:>9.4f}")

d_p1 = topn['<=512']['p@1'] - topn['>2048']['p@1']
d_p3 = topn['<=512']['p@3'] - topn['>2048']['p@3']
d_p5 = topn['<=512']['p@5'] - topn['>2048']['p@5']
print(f"\nB0→B3 하락:  P@1 {-d_p1*100:+.1f}pt  ·  P@3 {-d_p3*100:+.1f}pt  ·  P@5 {-d_p5*100:+.1f}pt")
print("최장 문서에서도 정답이 top-5에 98.1%·top-3에 96.4% 잔존 — 표현 붕괴가 아니라 top-1/임계 배치의 미끄러짐")

bin              n      P@1      P@3      P@5
<=512         2737   0.9098   0.9803   0.9865
512-1024      4369   0.9062   0.9821   0.9892


1024-2048     1996   0.8913   0.9734   0.9835
>2048          477   0.8616   0.9644   0.9811

B0→B3 하락:  P@1 -4.8pt  ·  P@3 -1.6pt  ·  P@5 -0.5pt
최장 문서에서도 정답이 top-5에 98.1%·top-3에 96.4% 잔존 — 표현 붕괴가 아니라 top-1/임계 배치의 미끄러짐


## 판정 — GPU label-aware attention 게이트

세 결과를 종합해 8192 헤드 교체 실험의 기대 이득 상한을 경계짓고 JSON에 저장한다.

In [9]:
verdict = (
 "장문 열화는 (1) 카디널리티와 독립이나(k>=2 비율 길이 무관 0.144→0.131, k==1에서도 P@1 0.9098→0.8616 생존) "
 "(2) 표현 붕괴가 아니다 — 최장 문서에서도 정답이 top-5 98.1%·top-3 96.4% 잔존(P@5 -0.5pt vs P@1 -4.8pt). "
 "정보는 풀링을 통과해 랭킹에 살아 있고 미끄러지는 건 top-1/임계 배치(결정층)다. "
 "8192 창이 장문 컨텍스트 가치를 이미 회수했고(exp1-exp2 B3 +2.64pt), 길이-조건부 캘리브레이션은 거의 소진(오라클 +0.08~0.24pt, ADR-0006). "
 "label-aware attention이 겨냥하는 표현 헤드룸은 R-Prec 갭(최장 5%에서 ~4pt)으로 상한이 얇고 본질적 난이도(넓은 범위의 긴 청구항)와 교락된다. "
 "확정(풀링 헤드가 R-Prec 갭을 회수하는가)은 8192 hidden-state 덤프가 필요하나, 코퍼스 micro 상한은 <~0.5pt로 경계지어진다 — 기대 이득이 얇은 레버."
)

result = {
    "num_labels": NUM, "tau": TAU, "split": "test",
    "models": MODELS,
    "degradation_curve": curve,          # 길이대별 exp1/exp2 지표
    "cardinality_by_bin": kx,            # 길이별 k 구성
    "k1_by_bin": k1, "k2_by_bin": k2,    # k 슬라이스별 저하
    "topN_k1_by_bin": topn,             # 표현 vs 결정 판별
    "verdict": verdict,
}
fp = OUT / "longdoc_probe_test.json"
fp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", fp)
print("\n" + verdict)

saved: C:\workspace\patent_disc\output\longdoc_probe_test.json

장문 열화는 (1) 카디널리티와 독립이나(k>=2 비율 길이 무관 0.144→0.131, k==1에서도 P@1 0.9098→0.8616 생존) (2) 표현 붕괴가 아니다 — 최장 문서에서도 정답이 top-5 98.1%·top-3 96.4% 잔존(P@5 -0.5pt vs P@1 -4.8pt). 정보는 풀링을 통과해 랭킹에 살아 있고 미끄러지는 건 top-1/임계 배치(결정층)다. 8192 창이 장문 컨텍스트 가치를 이미 회수했고(exp1-exp2 B3 +2.64pt), 길이-조건부 캘리브레이션은 거의 소진(오라클 +0.08~0.24pt, ADR-0006). label-aware attention이 겨냥하는 표현 헤드룸은 R-Prec 갭(최장 5%에서 ~4pt)으로 상한이 얇고 본질적 난이도(넓은 범위의 긴 청구항)와 교락된다. 확정(풀링 헤드가 R-Prec 갭을 회수하는가)은 8192 hidden-state 덤프가 필요하나, 코퍼스 micro 상한은 <~0.5pt로 경계지어진다 — 기대 이득이 얇은 레버.
